# Day 4

## Tokenizing with code

In [2]:
import tiktoken 
encoding = tiktoken.encoding_for_model("gpt-4.1-mini")
tokens = encoding.encode("Hi My name is Zulqarnain")

In [8]:
print(tokens)

[12194, 3673, 1308, 382, 124632, 80, 1978, 524]


In [10]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} -> {token_text}")

12194 -> Hi
3673 ->  My
1308 ->  name
382 ->  is
124632 ->  Zul
80 -> q
1978 -> arn
524 -> ain


In [18]:
encoding.decode([80])

'q'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

In [19]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("Error..............")
elif not api_key.startswith("sk-proj-"):
    print("Incorrect key")
else:
    print("API KEY found")

API KEY found


### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

In [ ]:
from openai import OpenAI
base_url ="http://localhost:11434/v1"
openai = OpenAI(base_url=base_url)
# response = openai.chat.completions.create(model="gemma3:270m", messages=[{"role": "user", "content": "Tell me your name"}])
# response.choices[0].message.content

'I am Gemma, an open-weights AI assistant.\n'

### A message to OpenAI is a list of dicts

In [43]:
messages = [
    {"role": "system", "content": "Your are a helpful assistant"},
    {"role": "user", "content": "Hi, I am Zulqarnain"}
]

In [44]:
repsonse = openai.chat.completions.create(model="llama3.2:latest", messages=messages)
repsonse.choices[0].message.content

"Hello Zulqarnain! It's nice to meet you. Is there something I can help you with or would you like to chat?"

### OK let's now ask a follow-up question

In [47]:
messages = [
    {"role": "system", "content": "Your are a helpful assistant"},
    {"role": "user", "content": "What is my name?"}
]

In [48]:
repsonse = openai.chat.completions.create(model="llama3.2:latest", messages=messages)
repsonse.choices[0].message.content

"I don't have any information about your personal details, including your name. Our conversation just started, and I'm happy to start assisting you with any questions or topics you'd like to discuss. Would you like to share your name with me?"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [49]:
messages = [
    {"role":"system", "content":"Your are a helpful assistant"},
    {"role": "user", "content": "My name is Zulqarnain"},
    {"role": "assistant", "content": "Hello! How can I help you today?"},
    {"role": "user", "content": "What is my name?"}
]

In [50]:
repsonse = openai.chat.completions.create(model="llama3.2:latest", messages=messages)
repsonse.choices[0].message.content

'You told me that your name is Zulqarnain.'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

